<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 12


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Item в C#, который будет представлять информацию о
товарах, которые могут быть заказаны или возвращены. На основе этого класса
разработать 2-3 производных класса, демонстрирующих принципы наследования и
полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и
методы, а также переопределены некоторые методы базового класса для
демонстрации полиморфизма.
Требования к базовому классу Item:
• Атрибуты: ID товара (ItemId), Название (Name), Цена (Price).
• Методы:
o
o GetDetails(): метод для получения детальной информации о товаре.
o CalculateDiscount(): метод для расчета скидки на товар.
o ApplyDiscount(decimal discount): метод для применения скидки к цене
товара.
Требования к производным классам:
1. ЕдиничныйТовар (SingleItem): Должен содержать дополнительные атрибуты,
такие как Единица измерения (UnitMeasure). Метод GetDetails() должен быть
переопределен для добавления информации о единице измерения товара.
2. ПакетныйТовар (PackageItem): Должен содержать дополнительные
атрибуты, такие как Количество единиц в пакете (QuantityPerPackage).
Метод CalculateDiscount() должен быть переопределен для учета количества
единиц в пакете при расчете скидки.
3. СпециальныйТовар (SpecialItem) (если требуется третий класс): Должен
содержать дополнительные атрибуты, такие как Дата истечения скидки
(DiscountExpirationDate). Метод ApplyDiscount() должен быть переопределен
для добавления информации о сроке действия скидки.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

public interface IShippable
{
    decimal CalculateShippingCost();
}

public interface IDiscountable
{
    void ApplyDiscount(decimal discount);
}

public class Item
{
    private string _itemId;
    private string _name;
    private decimal _price;
    private string _category;
    private string _manufacturer;
    private int _stockQuantity;
    private double _weight;
    private string _supplierCountry;
    private bool _isFragile;

    public delegate void ItemActionHandler(string message);
    public event ItemActionHandler OnItemAction;

    public string ItemId { get { return _itemId; } set { _itemId = value; } }
    public string Name { get { return _name; } set { _name = value; } }
    public decimal Price
    {
        get { return _price; }
        set
        {
            if (value >= 0) _price = value;
            else throw new ArgumentOutOfRangeException("Цена не может быть отрицательной");
        }
    }
    public string Category { get { return _category; } set { _category = value; } }
    public string Manufacturer { get { return _manufacturer; } set { _manufacturer = value; } }
    public int StockQuantity
    {
        get { return _stockQuantity; }
        set
        {
            if (value >= 0) _stockQuantity = value;
            else throw new ArgumentOutOfRangeException("Количество на складе не может быть отрицательным");
        }
    }
    public double Weight { get { return _weight; } set { _weight = value; } }
    public string SupplierCountry { get { return _supplierCountry; } set { _supplierCountry = value; } }
    public bool IsFragile { get { return _isFragile; } set { _isFragile = value; } }

    public Item(string itemId, string name, decimal price,
                string category, string manufacturer, int stockQuantity,
                double weight = 1.0, string supplierCountry = "Россия", bool isFragile = false)
    {
        ItemId = itemId;
        Name = name;
        Price = price;
        Category = category;
        Manufacturer = manufacturer;
        StockQuantity = stockQuantity;
        Weight = weight;
        SupplierCountry = supplierCountry;
        IsFragile = isFragile;
    }

    public void Restock(int amount)
    {
        if (amount > 0)
        {
            StockQuantity += amount;
            Console.WriteLine($"Товар '{Name}' пополнен на {amount}. Теперь на складе: {StockQuantity}");
            OnItemAction?.Invoke($"Товар '{Name}' пополнен. Текущий остаток: {StockQuantity}");
        }
        else
        {
            Console.WriteLine("Количество для пополнения должно быть больше 0.");
        }
    }

    public bool IsAvailable() => StockQuantity > 0;

    public string GetShortInfo() => $"{Name} ({Category}) от {Manufacturer}";

    public string GetShortInfo(bool includePrice)
    {
        if (includePrice) return $"{Name} ({Category}) от {Manufacturer} - Цена: {Price} руб.";
        return GetShortInfo();
    }

    public virtual string GetDetails()
    {
        return $"ID: {ItemId}, Название: {Name}, Цена: {Price}, " +
               $"Категория: {Category}, Производитель: {Manufacturer}, На складе: {StockQuantity}";
    }

    public virtual decimal CalculateDiscount() => Price * 0.05m;

    public virtual string GetHandlingInstructions()
    {
        return IsFragile ? "ВНИМАНИЕ: Хрупкий груз!" : "Стандартные условия перевозки.";
    }

    public virtual void ApplyDiscount(decimal discount)
    {
        Price -= discount;
        Console.WriteLine($"К товару '{Name}' применена скидка {discount}. Новая цена: {Price}");
        OnItemAction?.Invoke($"На товар '{Name}' применена скидка {discount}");
    }

    public void ApplyDiscount(double percentDiscount)
    {
        decimal discountAmount = Price * (decimal)(percentDiscount / 100);
        ApplyDiscount(discountAmount);
    }
}

public class SingleItem : Item
{
    private string unitMeasure;
    public string UnitMeasure { get { return unitMeasure; } set { unitMeasure = value; } }

    public SingleItem(string itemId, string name, decimal price, string unitMeasure,
                      string category, string manufacturer, int stockQuantity)
        : base(itemId, name, price, category, manufacturer, stockQuantity)
    {
        UnitMeasure = unitMeasure;
    }

    public override string GetDetails() => base.GetDetails() + $", Единица измерения: {UnitMeasure}";
}

public class PackageItem : Item
{
    private int quantity;
    public int Quantity
    {
        get { return quantity; }
        set
        {
            if (value > 0) quantity = value;
            else throw new ArgumentOutOfRangeException("Количество в пакете должно быть больше 0");
        }
    }

    public PackageItem(string itemId, string name, decimal price, int quantity,
                       string category, string manufacturer, int stockQuantity)
        : base(itemId, name, price, category, manufacturer, stockQuantity)
    {
        Quantity = quantity;
    }

    public override decimal CalculateDiscount()
    {
        decimal baseDiscount = base.CalculateDiscount();
        decimal volumeBonus = (Quantity / 5) * 0.01m * Price;
        return baseDiscount + volumeBonus;
    }

    public decimal CalculateDiscount(double extraPercent)
    {
        return CalculateDiscount() + Price * (decimal)(extraPercent / 100);
    }
}

public class SpecialItem : Item
{
    private DateTime discountExpirationDate;
    public DateTime DiscountExpirationDate { get { return discountExpirationDate; } set { discountExpirationDate = value; } }

    public SpecialItem(string itemId, string name, decimal price, DateTime discountExpirationDate,
                       string category, string manufacturer, int stockQuantity)
        : base(itemId, name, price, category, manufacturer, stockQuantity)
    {
        DiscountExpirationDate = discountExpirationDate;
    }

    public override void ApplyDiscount(decimal discount)
    {
        base.ApplyDiscount(discount);
        Console.WriteLine($"Скидка на товар '{Name}' действует до: {DiscountExpirationDate.ToShortDateString()}");
    }

    public override string GetHandlingInstructions()
    {
        return base.GetHandlingInstructions() + " Это подарочный товар, обращайтесь аккуратно.";
    }
}

public class DiscountedPackageItem : PackageItem, IShippable, IDiscountable
{
    public DiscountedPackageItem(string itemId, string name, decimal price, int quantity,
                                 string category, string manufacturer, int stockQuantity)
        : base(itemId, name, price, quantity, category, manufacturer, stockQuantity) { }

    decimal IShippable.CalculateShippingCost() => 200m + Quantity * 10m + (decimal)Weight * 5m;

    void IDiscountable.ApplyDiscount(decimal discount)
    {
        base.ApplyDiscount(discount);
        Console.WriteLine($"Товар '{Name}' участвует в акции. Страна-поставщик: {SupplierCountry}");
    }

    public override void ApplyDiscount(decimal discount) => ((IDiscountable)this).ApplyDiscount(discount);
}

public class ShippingService
{
    private readonly IShippable _shippableItem;
    public ShippingService(IShippable shippableItem) => _shippableItem = shippableItem;
    public void ProcessShipping() => Console.WriteLine($"Служба доставки рассчитала стоимость: {_shippableItem.CalculateShippingCost()} руб.");
}

public delegate bool ItemFilter(Item item);

public class Warehouse<T> where T : Item
{
    private List<T> _items = new List<T>();
    private Dictionary<string, T> _itemsDictionary = new Dictionary<string, T>();

    public void AddItem(T item)
    {
        _items.Add(item);
        if (!_itemsDictionary.ContainsKey(item.ItemId))
        {
            _itemsDictionary.Add(item.ItemId, item);
        }
    }

    public T GetItemById(string id)
    {
        _itemsDictionary.TryGetValue(id, out T foundItem);
        return foundItem;
    }

    public void ShowAllItems()
    {
        Console.WriteLine("Содержимое склада:");
        foreach (var item in _items) Console.WriteLine(item.GetShortInfo(true)); 
    }

    public void ShowFilteredItems(ItemFilter filter)
    {
        var filtered = _items.Where(i => filter(i)).ToList();
        Console.WriteLine("\n--- Результаты фильтрации ---");
        foreach (var item in filtered) Console.WriteLine(item.GetShortInfo(true));
    }
}

public class ShoppingCart
{
    private List<Item> _items = new List<Item>();
    public void AddItem(Item item) => _items.Add(item);

    public void Checkout()
    {
        foreach (var item in _items)
        {
            Console.WriteLine(item.GetDetails());
            Console.WriteLine($"Инструкции: {item.GetHandlingInstructions()}");
            Console.WriteLine(item.IsAvailable() ? "Товар в наличии." : "Товара нет на складе!");

            decimal discount = item.CalculateDiscount();
            Console.WriteLine($"Рассчитанная скидка: {discount}");
            item.ApplyDiscount(discount);

            if (item is IShippable shippable)
            {
                ShippingService shipper = new ShippingService(shippable);
                shipper.ProcessShipping();
            }
            Console.WriteLine(new string('-', 50));
        }
    }
}
Item[] shopInventory = new Item[]
{
    new SingleItem("1", "Молоко", 90, "литр", "Молочные продукты", "Молокозавод №1", 50),
    new PackageItem("2", "Упаковка газировки (12 шт)", 800, 12, "Напитки", "Кола-Компани", 20),
    new SpecialItem("3", "Новогодний набор", 1500, new DateTime(2027, 02, 15), "Подарки", "Фабрика Деда Мороза", 5),
    new DiscountedPackageItem("4", "Упаковка сока", 1200, 10, "Напитки", "Сок-Завод", 15)
};
shopInventory[3].Weight = 5.5;
shopInventory[3].IsFragile = true;
shopInventory[3].SupplierCountry = "Испания";
foreach (var item in shopInventory)
{
    item.OnItemAction += (msg) => Console.WriteLine($"[Событие]: {msg}");
}
ShoppingCart cart = new ShoppingCart();
foreach (var item in shopInventory) cart.AddItem(item);
Console.WriteLine(new string ('-', 50));
cart.Checkout();
Console.WriteLine("Перегрузка");
Item testItem = shopInventory[0];
Console.WriteLine(testItem.GetShortInfo()); 
Console.WriteLine(testItem.GetShortInfo(true)); 
Console.WriteLine("Перегруженный метод ApplyDiscount");
testItem.ApplyDiscount(10.0);
Console.WriteLine("Перекрытие");
Item single = shopInventory[0];
Item special = shopInventory[2];
Console.WriteLine(single.GetDetails()); 
Console.WriteLine(special.GetDetails()); 
Warehouse<Item> mainWarehouse = new Warehouse<Item>();
mainWarehouse.AddItem(shopInventory[0]);
mainWarehouse.AddItem(shopInventory[1]);
mainWarehouse.AddItem(shopInventory[2]);
mainWarehouse.ShowAllItems();
Console.WriteLine("Поиск по ID через словарь:");
var found = mainWarehouse.GetItemById("2");
if (found != null) Console.WriteLine($"Найден: {found.Name}");
Console.WriteLine("Фильтрация товаров дороже 1000 руб. (Делегат):");
mainWarehouse.ShowFilteredItems(item => item.Price > 1000);
Console.WriteLine("Уникальные категории товаров (HashSet):");
HashSet<string> categories = new HashSet<string>();
foreach (var item in shopInventory) categories.Add(item.Category);
foreach (var cat in categories) Console.WriteLine($"- {cat}");
Console.WriteLine("Очередь на пополнение (остаток < 10):");
Queue<Item> restockQueue = new Queue<Item>();
foreach (var item in shopInventory.Where(i => i.StockQuantity < 10)) restockQueue.Enqueue(item);
while (restockQueue.Count > 0)
{
    var itemToRestock = restockQueue.Dequeue();
    Console.WriteLine($"Требует пополнения: {itemToRestock.Name} (остаток: {itemToRestock.StockQuantity})");
}
Console.WriteLine("Демонстрация события (пополнение товара):");
shopInventory[0].Restock(30);

--------------------------------------------------
ID: 1, Название: Молоко, Цена: 90, Категория: Молочные продукты, Производитель: Молокозавод №1, На складе: 50, Единица измерения: литр
Инструкции: Стандартные условия перевозки.
Товар в наличии.
Рассчитанная скидка: 4.50
К товару 'Молоко' применена скидка 4.50. Новая цена: 85.50
[Событие]: На товар 'Молоко' применена скидка 4.50
--------------------------------------------------
ID: 2, Название: Упаковка газировки (12 шт), Цена: 800, Категория: Напитки, Производитель: Кола-Компани, На складе: 20
Инструкции: Стандартные условия перевозки.
Товар в наличии.
Рассчитанная скидка: 56.00
К товару 'Упаковка газировки (12 шт)' применена скидка 56.00. Новая цена: 744.00
[Событие]: На товар 'Упаковка газировки (12 шт)' применена скидка 56.00
--------------------------------------------------
ID: 3, Название: Новогодний набор, Цена: 1500, Категория: Подарки, Производитель: Фабрика Деда Мороза, На складе: 5
Инструкции: Стандартные условия перевозки